<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/lab4_with_pipieline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 - Project 2: Machine Learning with PySpark MLIB (with pipeline)

## Project Overview
In this second project, we implement the classification task using an MLIB Pipeline. A pipeline chains multiple Transformers and an Estimator together into a single workflow.

### Why use a Pipeline?
1. Prevents Data Leakage: Preprocessing before splitting (as in project 1) can subtly expose infromation from the test set to the model.
2. Reproducibility & Clean Code: Instead of calling '.fit()' and '.transform()' repeatedly on individual columns, the entire end-to-end pipeline is trained with a single call to 'pipeline.fit(train_df)'.
3. Seamless Inference: The resulting 'PipelineModel' can directly transform raw test data without manual step-by-step feature transformations.

In [ ]:
# STEP 1: install PySpark and download the dataset

!pip install pyspark

!wget -q -O bank.csv https://raw.githubusercontent.com/selva86/datasets/master/bank-full.csv


In [ ]:
# STEP 2: initalize SparkSession

from pyspark.sql import SparkSession

# initialize a new SparkSession for project 2
spark = SparkSession.builder.appName("BankMarketing_With_Pipeline").getOrCreate()


In [ ]:
# STEP 3: ingestion and train/test split (prior to transformations)

from pyspark.sql.functions import col

# read dataset with semicolon separator and schema inference
df = spark.read.format("csv").option("header", "true").option("sep", ";").option("inferSchema", "true").load("bank.csv")

# select relevant columns, cast numeric fields, and drop null rows
dataset = df.select(
    col("age").cast("float"),
    col("job"),
    col("marital"),
    col("education"),
    col("duration").cast("float"),
    col("campaign").cast("float"),
    col("y"),
).dropna(how="any")

# split raw data into train and test sets before applying any feature indexing
(train_df, test_df) = dataset.randomSplit([0.8, 0.2], seed=42)
print("Training Dataset Count: ", train_df.count())
print("Test Dataset Count: ", test_df.count())


Training Dataset Count:  32977
Test Dataset Count:  8211


In [ ]:
# STEP 4: construct the MLlib Pipeline

# import Pipeline and all required MLlib components
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

# stage 1-4: define StringIndexer transformers for categorical features and label
job_indexer = StringIndexer(inputCol="job", outputCol="job_index", handleInvalid="keep")
marital_indexer = StringIndexer(inputCol="marital", outputCol="marital_index", handleInvalid="keep")
education_indexer = StringIndexer(inputCol="education", outputCol="education_index", handleInvalid="keep")
label_indexer = StringIndexer(inputCol="y", outputCol="label")

# stage 5: define VectorAssembler transformers to aggregate features into one vector
feature_cols = ["age", "duration", "campaign", "job_index", "marital_index", "education_index"]
vector_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# stage 6. define the Random Forest estimator
rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=5)

# assmble all stages sequentially into a Pipeline
pipeline = Pipeline(stages=[
    job_indexer,
    marital_indexer,
    education_indexer,
    label_indexer,
    vector_assembler,
    rf_classifier
])



In [ ]:
# STEP 5: fit pipeline model and generate predictions

# fit the entire pipeline on training data in one step
pipeline_model = pipeline.fit(train_df)

# transform both training and test data using the fitted PipelineModel
train_predictions = pipeline_model.transform(train_df)
test_predictions = pipeline_model.transform(test_df)

# inspect predictions on unseen test data
test_predictions.select("y", "label", "features", "prediction", "prediction").show(5, truncate=False)



+---+-----+-----------------------------+----------+----------+
|y  |label|features                     |prediction|prediction|
+---+-----+-----------------------------+----------+----------+
|no |0.0  |[18.0,75.0,1.0,10.0,1.0,1.0] |0.0       |0.0       |
|no |0.0  |[19.0,43.0,3.0,10.0,1.0,2.0] |0.0       |0.0       |
|yes|1.0  |[19.0,126.0,4.0,10.0,1.0,2.0]|0.0       |0.0       |
|no |0.0  |[19.0,72.0,4.0,10.0,1.0,6.0] |0.0       |0.0       |
|yes|1.0  |[19.0,205.0,1.0,10.0,1.0,6.0]|0.0       |0.0       |
+---+-----+-----------------------------+----------+----------+
only showing top 5 rows


In [ ]:
# STEP 6: model evaluation and comparison

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# evaluate accuracy on train and test sets
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
train_acc = evaluator_acc.evaluate(train_predictions)
test_acc = evaluator_acc.evaluate(test_predictions)

# evaluate weighted F1-score on train and test sets
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")
train_f1 = evaluator_f1.evaluate(train_predictions)
test_f1 = evaluator_f1.evaluate(test_predictions)

# display final Pipeline metrics
print("=== Results with Pipeline ===")
print("Training Accuracy: ", train_acc)
print("Test Accuracy: ", test_acc)
print("Training F1-Score: ", train_f1)
print("Test F1-Score: ", test_f1)


=== Results with Pipeline ===
Training Accuracy:  0.8934712072050217
Test Accuracy:  0.8894166362197052
Training F1-Score:  0.8557285273380532
Test F1-Score:  0.8498865871648326
